In [1]:
import sys
sys.path.append("..")

In [2]:
from src.ingest import load_faq_data

In [3]:
documents = load_faq_data()
len(documents)

1406

In [4]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [5]:
documents_llm = []
for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)

len(documents_llm)

144

In [6]:
documents = documents_llm

In [7]:
doc = documents[0]
print(doc['id'])
print(doc['question'])
print(doc['answer'])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


### Generating questions with structured output

In [8]:
# We want the output as a list of strings, so we define that structure with a Pydantic model:

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The instructions for the LLM:

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called responses.create and read response.output_text. For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

The parsed object is available in response.output_parsed:

In [15]:
response.output_parsed.questions

['I just found this course — can I still join it now, or is it too late?',
 'If I start the course late, will I still be able to get a certificate?',
 'Is it okay to enroll after the course has already started?',
 'What do I need to do if I want a certificate after joining late?',
 'Can I still take part in the course, and does the project submission deadline affect the certificate?']

We can access the list directly:

In [16]:
result = response.output_parsed
print(result.questions)

['I just found this course — can I still join it now, or is it too late?', 'If I start the course late, will I still be able to get a certificate?', 'Is it okay to enroll after the course has already started?', 'What do I need to do if I want a certificate after joining late?', 'Can I still take part in the course, and does the project submission deadline affect the certificate?']


In [17]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

### Reusable utilities

We'll need this pattern again in other evaluation sections today, so
we put it in a reusable helper.

It contains helper functions we'll reuse in this module:

- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

In [19]:
from src.evaluation_utils import llm_structured

In [20]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'Is it too late to start the course now?', 'If I join the course after it already started, can I still get a certificate?', 'What do I need to do to qualify for a certificate if I’m joining late?', 'Can I participate in the course now, and how does the project submission deadline affect the certificate?']


In [21]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=298)

### Tracking cost

In [23]:
from src.evaluation_utils import calc_price

In [24]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00040950000000000003,
 'total_cost': 0.00056475}

Now convert these questions into ground truth records:

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course after it already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Can I participate in the course now, and how does the project submission deadline affect the certificate?',
  'document': '74eb249bbf'}]

Each record has two fields:

- `question`: the question generated by the LLM
- `document`: the ID of the FAQ document that should answer the question

The `document` field connects the generated question to the document
that contains the answer. Later, when we evaluate search, we'll ask the
search engine the generated question. Then we'll check if it retrieves
the document with this ID.

We now know how to generate and store questions for one document. In
the next lesson, we'll run this for 

In [27]:
import pandas as pd

In [28]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start the course now?,74eb249bbf
2,"If I join the course after it already started,...",74eb249bbf
3,What do I need to do to qualify for a certific...,74eb249bbf
4,"Can I participate in the course now, and how d...",74eb249bbf


### Generating Ground Truth for All Documents

We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.

For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a `Questions` object
- create one ground truth record for each generated question

Each record contains the generated question and the ID of the document
that should answer the question.

When we send many requests, one of them might fail. We don't want the
entire batch to fail because of one temporary error.

In [29]:
from src.evaluation_utils import llm_structured_retry

`llm_structured` makes one structured-output call. `llm_structured_retry`
wraps the same call in a retry loop. If one request fails because of a
temporary API or network issue, it waits briefly and tries again.

In [ ]:
def generate_ground_truth(doc):
    
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [31]:
from tqdm.auto import tqdm

In [32]:
ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

This works, but it runs one LLM call after another. Running it for all
documents this way would take too long.

### Parallel processing

Running the calls one after another wastes most of the time waiting on
the network. Each request just sits there until OpenAI responds, so we
can fire several at once and wait on them together. We process the
documents in parallel and track progress while the requests run.

In [33]:
from concurrent.futures import ThreadPoolExecutor
from src.evaluation_utils import map_progress

This submits one job per document, updates the progress bar when a job
finishes, and collects the results

In [34]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/144 [00:00<?, ?it/s]

`generate_ground_truth` returns two things for each document: the
generated records and the token usage.

Split those into separate lists:

In [35]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

720

Calculate the total cost:

In [36]:
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.11203875000000002

We'll calculate total cost several times in this module, so the utility
file has a helper for it:

In [37]:
from src.evaluation_utils import calc_total_price

calc_total_price(usages)

0.11203875000000002

Create a dataframe so we can look at the records as a table and save
them as a CSV file.

In [38]:
df_ground_truth = pd.DataFrame(ground_truth)
print(df_ground_truth.shape)
df_ground_truth.head()

(720, 2)


,question,document
0,I just found this course — is it still okay to...,74eb249bbf
1,Can I still enroll if I discovered the course ...,74eb249bbf
2,"If I join the course late, can I still get a c...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Is it fine to take the course after it has beg...,74eb249bbf


Because we generated the questions from specific documents, we know
which document is correct for each question. We now have the ground
truth we need for evaluation.

Save it for later use:

In [40]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)